In [11]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, lamd=1e-2, epsilon=1e-8):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr, "beta1": beta1, "beta2": beta2, "lamd": lamd, "epsilon": epsilon}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        
        for group in self.param_groups: # 这个 param_groups 是参数组，代表模型不同模块的参数可以有不同的学习率
            lr = group["lr"]  # Get the learning rate.
            beta1 = group["beta1"] # 一阶动量平滑系数
            beta2 = group["beta2"] # 二阶动量平滑系数
            lamd = group["lamd"] # weight decay
            epsilon = group["epsilon"] # 分母
            
            
            for p in group["params"]:
                if p.grad is None:
                    continue
                
                # 1. 取state
                state = self.state[p]  # state 是一个map，从参数param映射到字典
                t = state.get("t", 1)  # Get iteration number from the state, or initial value.
                m = state.get("m", torch.zeros_like(p))
                v = state.get("v", torch.zeros_like(p))
                grad = p.grad.data  # Get the gradient of loss with respect to p.
                
                # 2. 更新动量
                m = beta1 * m + (1 - beta1) * grad
                v = beta2 * v + (1 - beta2) * grad**2
                
                # 3. 更新参数
                alpha_t = lr * math.sqrt(1 - beta2**t) / (1 - beta1**t)
                p.data -= alpha_t * m / (torch.sqrt(v) + epsilon)
                
                # 4. weight decay(正则化)
                p.data -= lr * lamd * p.data
                
                # 5. 更新状态
                state["t"] = t + 1
                state["m"] = m
                state["v"] = v
        
        return loss


In [12]:
# 初始化参数和优化器
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
target = torch.diag(torch.ones(10))
opt = AdamW([weights], lr=1)

# 训练循环
for t in range(100):
    opt.zero_grad()  # Reset the gradients for all learnable parameters.
    loss = ((weights - target)**2).mean()  # Compute a scalar loss value.
    print(f"Step {t:3d} | Loss: {loss.cpu().item():.6f}")
    loss.backward()  # Run backward pass, which computes gradients.
    opt.step()  # Run optimizer step.

Step   0 | Loss: 22.611902
Step   1 | Loss: 15.679573
Step   2 | Loss: 10.611707
Step   3 | Loss: 7.119635
Step   4 | Loss: 4.796460
Step   5 | Loss: 3.263150
Step   6 | Loss: 2.266057
Step   7 | Loss: 1.646630
Step   8 | Loss: 1.296707
Step   9 | Loss: 1.132339
Step  10 | Loss: 1.092150
Step  11 | Loss: 1.131117
Step  12 | Loss: 1.210458
Step  13 | Loss: 1.293307
Step  14 | Loss: 1.349625
Step  15 | Loss: 1.362854
Step  16 | Loss: 1.329856
Step  17 | Loss: 1.255766
Step  18 | Loss: 1.149107
Step  19 | Loss: 1.020117
Step  20 | Loss: 0.880577
Step  21 | Loss: 0.742451
Step  22 | Loss: 0.615481
Step  23 | Loss: 0.505617
Step  24 | Loss: 0.415242
Step  25 | Loss: 0.344216
Step  26 | Loss: 0.290544
Step  27 | Loss: 0.250821
Step  28 | Loss: 0.221078
Step  29 | Loss: 0.197859
Step  30 | Loss: 0.178991
Step  31 | Loss: 0.163624
Step  32 | Loss: 0.151640
Step  33 | Loss: 0.142848
Step  34 | Loss: 0.136451
Step  35 | Loss: 0.131088
Step  36 | Loss: 0.125234
Step  37 | Loss: 0.117662
Step  38 

Step   0 | Loss: 22.191753
Step   1 | Loss: 14.202724
Step   2 | Loss: 10.469640
Step   3 | Loss: 8.191376
Step   4 | Loss: 6.635015
Step   5 | Loss: 5.501187
Step   6 | Loss: 4.639522
Step   7 | Loss: 3.964604
Step   8 | Loss: 3.423747
Step   9 | Loss: 2.982464
Step  10 | Loss: 2.617139
Step  11 | Loss: 2.311017
Step  12 | Loss: 2.051867
Step  13 | Loss: 1.830546
Step  14 | Loss: 1.640083
Step  15 | Loss: 1.475069
Step  16 | Loss: 1.331250
Step  17 | Loss: 1.205232
Step  18 | Loss: 1.094280
Step  19 | Loss: 0.996166
Step  20 | Loss: 0.909058
Step  21 | Loss: 0.831441
Step  22 | Loss: 0.762047
Step  23 | Loss: 0.699813
Step  24 | Loss: 0.643840
Step  25 | Loss: 0.593363
Step  26 | Loss: 0.547729
Step  27 | Loss: 0.506376
Step  28 | Loss: 0.468821
Step  29 | Loss: 0.434644
Step  30 | Loss: 0.403482
Step  31 | Loss: 0.375016
Step  32 | Loss: 0.348967
Step  33 | Loss: 0.325091
Step  34 | Loss: 0.303172
Step  35 | Loss: 0.283021
Step  36 | Loss: 0.264467
Step  37 | Loss: 0.247362
Step  38 

In [13]:
weights

Parameter containing:
tensor([[ 9.8781e-01,  3.9923e-03, -8.0092e-04,  8.1853e-03,  5.1639e-03,
         -9.0995e-03,  3.9506e-03, -1.6759e-03, -6.6894e-03,  8.8328e-03],
        [ 4.1524e-03,  9.9478e-01,  5.8343e-03, -9.5774e-04, -2.5874e-03,
         -1.7906e-03, -3.7638e-03,  2.5411e-03,  5.4774e-03, -1.3752e-02],
        [ 1.8899e-03,  2.3447e-03,  9.9749e-01, -6.1937e-03, -2.5829e-03,
          4.4653e-03, -9.7612e-04, -2.0334e-03,  2.1447e-03, -6.1094e-03],
        [-5.7304e-03,  1.6007e-02,  2.1957e-02,  1.0020e+00, -4.8330e-03,
         -6.9518e-05, -2.5398e-02, -9.8272e-03,  5.5484e-03,  4.8016e-04],
        [ 1.7034e-03, -6.6661e-03,  7.2329e-03,  4.3086e-03,  9.9795e-01,
          1.5342e-02, -4.5995e-03, -2.0853e-03,  1.0800e-02,  1.9730e-02],
        [-4.4069e-03, -6.3150e-03,  4.2668e-04, -1.8794e-02,  6.2847e-03,
          1.0004e+00, -3.4874e-03,  1.6120e-03, -6.5959e-04,  9.7982e-03],
        [ 1.5322e-04,  8.0497e-03,  1.0867e-03,  2.1172e-03,  1.6109e-03,
          